# Delta策略单日持仓生成与回测案例展示

本notebook展示如何使用delta模块进行单日持仓生成和回测。

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('/home/jovyan/base_demo')
import base_tool

import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

## 导入delta模块

In [ ]:
import sys
sys.path.append('/home/jovyan/work/tactics_demo')
from delta import *

## 1. 设置策略参数

In [ ]:
# 设置标的和日期
instrument_id = '518880'
trade_ymd = '20260319'

# 策略参数配置
param_dict = {
    'instrument_id': instrument_id,
    'trade_ymd': trade_ymd,
    'name': 'delta_v1_demo',
    'stride': 1,

    'short_window': 60,      # 短期窗口
    'long_window': 300,      # 长期窗口
    'y_window': 300,         # 标签窗口

    'open_threshold': 2,     # 开仓阈值
    'open_confidence': 0,    # 开仓置信度
    'close_confidence': 0.2, # 平仓置信度
    'standard_num': 1000,    # 标准数量

    'k_pct': 1,
    'k_up': 3,
    'k_down': 3,
}
param_dict['x_window'] = max(param_dict['short_window'], param_dict['long_window'])

## 2. 训练模型（可选）

In [ ]:
# 获取交易日数据
trade_dates = get_trade_dates()
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

# 分割数据集
train_dates, valid_dates, test_dates = split_dates(trade_dates)

# 生成训练数据
print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")

# 训练模型
print("训练模型...")
model = train_model(X_train, y_train, X_valid, y_valid, param_dict)

# 保存模型（可选）
model_filename = f"delta_xgboost_model_{instrument_id}.joblib"
save_model(model, model_filename)
print(f"模型已保存到: {model_filename}")

## 3. 加载预训练模型（推荐）

In [ ]:
# 加载预训练模型
model_filename = f"delta_xgboost_model_{instrument_id}.joblib"
model = load_model(model_filename)
print(f"模型已加载: {model_filename}")

## 4. 创建策略实例

In [ ]:
# 创建策略实例
strategy = StrategyDemo(model, param_dict)
print(f"策略已创建: {strategy.name}")
print(f"策略参数: {param_dict}")

## 5. 单日持仓生成

In [ ]:
# 加载当日数据
snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
print(f"加载 {trade_ymd} 数据，共 {len(snap_list)} 个snapshot")

# 生成持仓信号
position_dict = {}
for snap in snap_list:
    strategy.on_snap(snap)
    position_dict[snap['time_mark']] = strategy.position_last

# 转换为DataFrame查看
position_df = pd.DataFrame(list(position_dict.items()), columns=['time_mark', 'position'])
position_df['position'] = position_df['position'].astype(int)

print("\n持仓信号统计:")
print(f"总snapshot数: {len(position_df)}")
print(f"做多次数: {(position_df['position'] == 1).sum()}")
print(f"做空次数: {(position_df['position'] == -1).sum()}")
print(f"空仓次数: {(position_df['position'] == 0).sum()}")

# 查看前10个持仓信号
print("\n前10个持仓信号:")
print(position_df.head(10))

## 6. 单日回测

In [ ]:
# 导入回测工具
import sys
sys.path.append('/home/jovyan/work/tactics_demo/tools')
from backtest_quick import backtest_quick

# 执行单日回测
profit_df = backtest_quick(
    instrument_id=instrument_id,
    trade_ymd=trade_ymd,
    strategy_name=strategy.name,
    position_dict=position_dict,
    remake=True  # 强制重新计算
)

if profit_df is not None and len(profit_df) > 0:
    print(f"回测完成，当日盈亏: {profit_df['profits'].iloc[-1]:.2f}")
    print(f"交易次数: {profit_df['trade'].iloc[-1]}")
    
    # 显示回测结果
    print("\n回测结果摘要:")
    print(profit_df[['time_mark', 'position', 'profits', 'trade']].tail(10))
    
    # 绘制盈亏曲线
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 6))
    
    # 子图1: 盈亏曲线
    plt.subplot(2, 1, 1)
    plt.plot(profit_df['time_mark'], profit_df['profits'])
    plt.title(f'{trade_ymd} 盈亏曲线')
    plt.xlabel('时间戳')
    plt.ylabel('累计盈亏')
    plt.grid(True)
    
    # 子图2: 持仓变化
    plt.subplot(2, 1, 2)
    plt.step(profit_df['time_mark'], profit_df['position'], where='post')
    plt.title('持仓变化')
    plt.xlabel('时间戳')
    plt.ylabel('持仓 (-1/0/1)')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("回测失败或无数据")

## 7. 多日回测（批量处理）

In [ ]:
# 导入多日回测工具
from multi_day_backtest import backtest_multi_days, backtest_summary

# 设置回测日期范围
start_ymd = '20251201'
end_ymd = '20251231'

# 执行多日回测
result_df = backtest_multi_days(
    instrument_id=instrument_id,
    start_ymd=start_ymd,
    end_ymd=end_ymd,
    StrategyClass=StrategyDemo,
    model=model,
    param_dict=param_dict,
    official=False
)

if result_df is not None:
    # 生成回测摘要
    summary = backtest_summary(result_df)
    print("多日回测摘要:")
    print(summary)
    
    # 计算累计收益
    result_df['cumulative_profit'] = result_df['profits'].cumsum()
    
    # 绘制累计收益图
    plt.figure(figsize=(12, 6))
    
    plt.subplot(2, 1, 1)
    plt.bar(result_df['trade_ymd'], result_df['profits'])
    plt.title('每日盈亏')
    plt.xlabel('交易日')
    plt.ylabel('盈亏')
    plt.xticks(rotation=45)
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    plt.plot(result_df['trade_ymd'], result_df['cumulative_profit'], marker='o')
    plt.title('累计收益曲线')
    plt.xlabel('交易日')
    plt.ylabel('累计收益')
    plt.xticks(rotation=45)
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 保存结果
    output_dir = f'/home/jovyan/work/tactics_demo/delta/backtest_result/{instrument_id}_{strategy.name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    os.makedirs(output_dir, exist_ok=True)
    result_df.to_csv(f'{output_dir}/daily_results.csv', index=False)
    print(f"结果已保存到: {output_dir}")
else:
    print("多日回测失败或无数据")

## 8. 核心调用示例总结

In [ ]:
'''
核心调用步骤总结:

1. 导入模块:
   from delta import StrategyDemo, create_feature, create_y
   from delta import samples_from_dates, train_model, load_model
   from backtest_quick import backtest_quick
   from multi_day_backtest import backtest_multi_days

2. 设置参数:
   param_dict = {
       'instrument_id': '518880',
       'trade_ymd': '20260319',
       'name': 'delta_strategy',
       'short_window': 60,
       'long_window': 300,
       ...
   }

3. 创建策略实例:
   strategy = StrategyDemo(model, param_dict)

4. 单日持仓生成:
   snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
   position_dict = {}
   for snap in snap_list:
       strategy.on_snap(snap)
       position_dict[snap['time_mark']] = strategy.position_last

5. 单日回测:
   profit_df = backtest_quick(instrument_id, trade_ymd, strategy.name, position_dict, remake=True)

6. 多日回测:
   result_df = backtest_multi_days(instrument_id, start_ymd, end_ymd, StrategyDemo, model, param_dict)
'''